# 08 — Model Deployment and Batch Inference
## Intelligent Customer Analytics Platform

Purpose: Load the trained CLV model from MLflow,
run batch inference on all 93,256 customers,
generate prediction scores, and save results
as a Gold Delta table for business consumption.

Production Pattern:

1. Load the selected model artifact from an MLflow experiment run
2. Run batch predictions on customer data
3. Generate high-value customer probability scores
4. Classify customers into business tiers
5. Save the scored customer table to the Gold Delta layer
6. Demonstrate single-record inference

In [0]:
# install required libraries

%pip install xgboost mlflow scikit-learn

In [0]:
# restart kernel after installation

dbutils.library.restartPython()

In [0]:
# verify libraries and set up paths

import mlflow
import xgboost as xgb
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

gold_path   = "/Volumes/workspace/default/olist_raw_data/gold"

print(f"mlflow version  : {mlflow.__version__}")
print(f"xgboost version : {xgb.__version__}")
print(f"spark version   : {spark.version}")
print("environment ready for model deployment")

In [0]:
# load the trained CLV model from MLflow experiment

experiment_path = "/Users/elitahazelgorimanikonda@gmail.com/CLV_Customer_Segmentation"

# get the best run from MLflow experiment
experiment = mlflow.get_experiment_by_name(experiment_path)

if experiment is None:
    raise ValueError(
        f"MLflow experiment was not found: {experiment_path}"
    )

experiment_id = experiment.experiment_id

runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.roc_auc DESC"]
)

if runs.empty:
    raise ValueError(
        "No model runs were found in the MLflow experiment."
    )

best_run_id  = runs.iloc[0]["run_id"]
best_roc_auc = runs.iloc[0]["metrics.roc_auc"]

print(f"experiment found  : {experiment_path}")
print(f"total runs        : {len(runs)}")
print(f"best run id       : {best_run_id}")
print(f"best roc auc      : {best_roc_auc:.4f}")

### V1 Reference Model

The V1 model is retained only to demonstrate 
the effect of directly derived RFM score features. 
It is not used as the final deployment model 
because those features create target leakage.

In [0]:
# load the best model from MLflow

model_uri = f"runs:/{best_run_id}/xgboost_clv_model"

print(f"loading model from: {model_uri}")

# load model as xgboost flavor
model = mlflow.xgboost.load_model(model_uri)

print("model loaded successfully!")
print(f"model type : {type(model)}")

In [0]:
# load all customers from gold layer for batch inference

gold_df = spark.read.format("delta") \
    .load(f"{gold_path}/rfm_features")

# convert to pandas for inference
inference_pdf = gold_df.select(
    "customer_unique_id",
    "recency",
    "frequency",
    "monetary",
    "customer_segment",
    "is_high_value"
).toPandas()

# fill nulls
inference_pdf = inference_pdf.fillna(0)

print(f"customers loaded for inference : {len(inference_pdf):,}")
print(f"\nsample data:")
print(inference_pdf.head(3))

In [0]:
# run batch inference on all 93,256 customers
# prepare all 7 features that model was trained on
# we need to add the score columns back
# recalculate them for inference
# calculate scores using same logic as training
inference_pdf["r_score"] = pd.qcut(
    inference_pdf["recency"].rank(method="first"), 
    5, labels=[5,4,3,2,1]
).astype(int)

inference_pdf["f_score"] = pd.qcut(
    inference_pdf["frequency"].rank(method="first"), 
    5, labels=[1,2,3,4,5]
).astype(int)

inference_pdf["m_score"] = pd.qcut(
    inference_pdf["monetary"].rank(method="first"), 
    5, labels=[1,2,3,4,5]
).astype(int)

inference_pdf["rfm_total_score"] = (
    inference_pdf["r_score"] + 
    inference_pdf["f_score"] + 
    inference_pdf["m_score"]
)

# now use all 7 features
X_inference = inference_pdf[[
    "recency", 
    "frequency", 
    "monetary",
    "r_score",
    "f_score", 
    "m_score",
    "rfm_total_score"
]]

# generate predictions and probability scores
predicted_class = model.predict(X_inference)
predicted_proba = model.predict_proba(X_inference)[:, 1]

# add predictions to dataframe
inference_pdf["predicted_high_value"]  = predicted_class
inference_pdf["clv_probability_score"] = predicted_proba.round(4)

# classify by probability score into business tiers
inference_pdf["clv_tier"] = inference_pdf["clv_probability_score"].apply(
    lambda x: "Platinum" if x >= 0.8 else
              "Gold"     if x >= 0.6 else
              "Silver"   if x >= 0.4 else
              "Bronze"
)

print("batch inference complete!")
print(f"total customers scored : {len(inference_pdf):,}")
print(f"\nCLV tier distribution:")
print(inference_pdf["clv_tier"].value_counts())
print(f"\nsample predictions:")
print(inference_pdf[[
    "customer_unique_id",
    "recency",
    "monetary",
    "clv_probability_score",
    "clv_tier"
]].head(5))

### V2 Deployment Candidate

The V2 model removes the directly derived RFM 
score columns and uses only Recency, Frequency, 
and Monetary values. All saved deployment 
predictions are generated using V2.

Note: V2 reduces direct feature leakage. 
The target remains related to RFM variables 
as it was derived from RFM segmentation.

In [0]:
# load v2 model with directly derived score features removed

runs_v2 = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.mlflow.runName = 'xgboost_clv_v2_no_leakage'",
    order_by=["metrics.roc_auc DESC"]
)

if not runs_v2.empty:
    best_run_id_v2 = runs_v2.iloc[0]["run_id"]
    model_uri_v2   = f"runs:/{best_run_id_v2}/xgboost_clv_v2"
    model_v2       = mlflow.xgboost.load_model(model_uri_v2)
    print(f"v2 model loaded: {model_uri_v2}")
else:
    available_runs = mlflow.search_runs(
        experiment_ids=[experiment_id]
    )[[
        "run_id",
        "tags.mlflow.runName",
        "metrics.roc_auc"
    ]]
    print(available_runs)
    raise ValueError(
        "Approved V2 model was not found. "
        "Batch inference cannot continue."
    )

In [0]:
# run batch inference using v2 model (raw RFM features only)

X_inference_v2 = inference_pdf[[
    "recency",
    "frequency", 
    "monetary"
]]

# generate predictions
predicted_class_v2 = model_v2.predict(X_inference_v2)
predicted_proba_v2 = model_v2.predict_proba(X_inference_v2)[:, 1]

# add to dataframe
inference_pdf["predicted_high_value"]  = predicted_class_v2
inference_pdf["clv_probability_score"] = predicted_proba_v2.round(4)

# assign business tiers based on probability
inference_pdf["clv_tier"] = inference_pdf["clv_probability_score"].apply(
    lambda x: "Platinum" if x >= 0.8 else
              "Gold"     if x >= 0.6 else
              "Silver"   if x >= 0.4 else
              "Bronze"
)

print("batch inference complete using v2 model!")
print(f"total customers scored : {len(inference_pdf):,}")
print(f"\nCLV tier distribution:")
print(inference_pdf["clv_tier"].value_counts())
print(f"\nprobability score stats:")
print(inference_pdf["clv_probability_score"].describe())
print(f"\ntop 5 highest value customers:")
print(inference_pdf.nlargest(5, "clv_probability_score")[[
    "customer_unique_id",
    "recency",
    "monetary",
    "clv_probability_score",
    "clv_tier"
]])

In [0]:
# save scored customers with full audit metadata

import uuid
from pyspark.sql import functions as F

# fix data types
inference_pdf["customer_unique_id"]    = inference_pdf["customer_unique_id"].astype(str)
inference_pdf["customer_segment"]      = inference_pdf["customer_segment"].astype(str)
inference_pdf["clv_tier"]              = inference_pdf["clv_tier"].astype(str)
inference_pdf["recency"]               = inference_pdf["recency"].astype(int)
inference_pdf["frequency"]             = inference_pdf["frequency"].astype(int)
inference_pdf["monetary"]              = inference_pdf["monetary"].astype(float)
inference_pdf["is_high_value"]         = inference_pdf["is_high_value"].astype(int)
inference_pdf["predicted_high_value"]  = inference_pdf["predicted_high_value"].astype(int)
inference_pdf["clv_probability_score"] = inference_pdf["clv_probability_score"].astype(float)

# generate unique batch id for this scoring run
scoring_batch_id = str(uuid.uuid4())

# convert to spark
scored_spark_df = spark.createDataFrame(
    inference_pdf[[
        "customer_unique_id",
        "recency",
        "frequency",
        "monetary",
        "customer_segment",
        "is_high_value",
        "predicted_high_value",
        "clv_probability_score",
        "clv_tier"
    ]]
)

# add audit metadata
scored_spark_df_final = scored_spark_df \
    .withColumn("model_run_id",
                F.lit(best_run_id_v2)) \
    .withColumn("model_name",
                F.lit("xgboost_clv_v2")) \
    .withColumn("model_uri",
                F.lit(model_uri_v2)) \
    .withColumn("scoring_batch_id",
                F.lit(scoring_batch_id)) \
    .withColumn("scored_at",
                F.current_timestamp()) \
    .withColumn("binary_classification_threshold",
                F.lit(0.5))

# save to gold delta layer
scored_spark_df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}/clv_scored_customers")

print("scored table saved with audit metadata")
print(f"scoring batch id : {scoring_batch_id}")
print(f"model run id     : {best_run_id_v2}")
print(f"model name       : xgboost_clv_v2")

In [0]:
# validate pandas and saved delta outputs match

scored_df = spark.read.format("delta") \
    .load(f"{gold_path}/clv_scored_customers")

pandas_distribution = (
    inference_pdf["clv_tier"]
    .value_counts()
    .sort_index()
    .to_dict()
)

delta_distribution = {
    row["clv_tier"]: row["count"]
    for row in (
        scored_df.groupBy("clv_tier")
        .count()
        .collect()
    )
}

print("Pandas tier distribution:")
print(pandas_distribution)

print("\nSaved Delta tier distribution:")
print(delta_distribution)

assert pandas_distribution == delta_distribution, (
    "Validation failed: tier counts do not match."
)

assert len(inference_pdf) == scored_df.count(), (
    "Validation failed: row counts do not match."
)

print("\nValidation successful: counts and distributions match!")

In [0]:
# single-record inference simulation

def predict_customer_value(
    recency: int,
    frequency: int,
    monetary: float
) -> dict:

    if recency < 0:
        raise ValueError("Recency cannot be negative.")
    if frequency < 0:
        raise ValueError("Frequency cannot be negative.")
    if monetary < 0:
        raise ValueError("Monetary value cannot be negative.")

    customer = pd.DataFrame([{
        "recency"  : int(recency),
        "frequency": int(frequency),
        "monetary" : float(monetary)
    }])

    prediction  = int(model_v2.predict(customer)[0])
    probability = float(
        model_v2.predict_proba(customer)[0, 1]
    )

    if probability >= 0.8:
        tier = "Platinum"
    elif probability >= 0.6:
        tier = "Gold"
    elif probability >= 0.4:
        tier = "Silver"
    else:
        tier = "Bronze"

    return {
        "predicted_high_value" : prediction,
        "clv_probability_score": round(probability, 4),
        "clv_tier"             : tier
    }

# test with two sample customers
print("Single-record inference simulation")
print("-" * 40)

print("\nRecent high spender:")
print(predict_customer_value(
    recency=45, frequency=4, monetary=850.0
))

print("\nInactive low spender:")
print(predict_customer_value(
    recency=400, frequency=1, monetary=50.0
))

In [0]:
# business summary report from scored customers

from pyspark.sql.functions import count, avg, round as spark_round

scored_df = spark.read.format("delta") \
    .load(f"{gold_path}/clv_scored_customers")

print("CLV Deployment — Business Summary Report")
print("=" * 50)

# tier distribution
print("\nCustomer CLV Tier Distribution:")
print("-" * 50)
scored_df.groupBy("clv_tier") \
    .agg(
        count("customer_unique_id").alias("customers"),
        spark_round(avg("monetary"), 2).alias("avg_spend"),
        spark_round(avg("clv_probability_score"), 4)
        .alias("avg_clv_score")
    ) \
    .orderBy("avg_clv_score", ascending=False) \
    .show()

# segment vs predicted comparison
print("\nActual vs Predicted High Value:")
print("-" * 50)
scored_df.groupBy("customer_segment", "clv_tier") \
    .count() \
    .orderBy("customer_segment", "clv_tier") \
    .show()

print("deployment summary complete!")

## Model Deployment Summary

### Model Details
- Model    : XGBoost Customer Value Classifier V2
- Source   : Selected MLflow experiment run
- Features : Recency, Frequency, Monetary
- Pattern  : Batch inference
- CLV Score: Predicted probability that a customer 
  belongs to the high-value class. Not a predicted 
  monetary CLV amount.
- Scoring  : 93,256 customers

### Batch Inference Results — Verified

| CLV Tier | Customers | Avg Spend | Avg CLV Score |
|----------|-----------|-----------|---------------|
| Platinum | 33,552    | R$ 226.00 | 0.9994        |
| Gold     | 124       | R$ 312.13 | 0.6910        |
| Silver   | 88        | R$ 128.08 | 0.5041        |
| Bronze   | 59,492    | R$ 130.91 | 0.0006        |

### Key Business Findings
- 33,552 customers classified as Platinum (approximately 36.0%)
- 59,492 customers classified as Bronze (approximately 63.8%)
- Gold and Silver together represent approximately 0.23%
- Gold customers had highest average historical spend
  in this scoring run (R$ 312.13)
- Platinum customers have average predicted 
  high-value probability of approximately 99.9%
- Tiers are based on predicted high-value probability,
  not historical spending alone. Therefore average 
  spending may not increase monotonically across tiers.

### Deployment Output
- Predictions saved to Gold Delta layer
- Path: /gold/clv_scored_customers
- Audit metadata included:
  model run ID, model name, model URI,
  scoring batch ID, timestamp,
  classification threshold

### Business Use
- Target Platinum customers for retention
- Identify Bronze customers with potential for targeted engagement campaigns
- Use probability scores for campaign prioritization

### Production Considerations
- Schedule with Databricks Workflows
- Monitor input and prediction distributions
- Retrain with fresh customer behavior data
- Use future spend as target for true CLV model
- Use Databricks Model Serving for real-time API